# B2-024 — Practice p22

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** core · **Time budget:** 65 minutes  
**Concepts:** open-ended-experiment-design

## Task

Run a budgeted experiment campaign on the inverse task: hypothesis → configurations → logged results → decision → one test call.

**Rules.**

* Before the first training run, define two strings: `HYPOTHESIS` (change, metric, direction, minimum effect, protocol, as in Session 2) and `DECISION_RULE` (how the logged validation numbers will choose the final configuration, including a tie or "not demonstrated" clause).
* Models: MLPs no wider than 64 hidden units, trained on `capstone_data.train_rows("inverse")` through a `fit(train_x, train_y)` method of your model class; **seam:** `fit` records `capstone_data.split_of("inverse", row)` for every row of `train_x`, and every value must be `"train"`. Preprocessing statistics come from `train_x` only.
* One `budget = capstone_data.StepBudget(2000)` wraps every optimizer of the campaign; at least **three** configurations and at least **two** seeds per configuration, with the same steps per run for every run; write the budget arithmetic in a comment before the first run.
* Every run appends a record `{"run", "config", "seed", "steps", "val_error", "seconds", "note"}` to a list `LOG`, where `val_error` is the mean validation position error.
* Apply `DECISION_RULE` by code to `LOG` (a function `decide(log)` that returns the chosen config name).

A suitable hypothesis family (you may choose another): raw versus log magnitudes versus log magnitudes with Gaussian input-noise augmentation (standard deviation `0.03` on the log features, fresh noise each step from the global generator).

**Evaluation protocol.** Fit only on rows from `capstone_data.train_rows("inverse")` (and any other rows this statement names), make every choice on `capstone_data.val_rows("inverse")`, and reach the locked test rows only through `capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="inverse")`, called exactly once, after all fitting and selection. Assert `capstone_data.test_call_count("inverse") == 0` immediately before that call and `== 1` after it.
Score the chosen configuration's seed-0 model from the campaign (do not train again).

**Write** a five-sentence campaign summary: what was tested, what the log shows (with per-seed numbers), which differences are demonstrated, the decision, and the next experiment you would run with a further 2,000 steps.

**Certify.** `budget.used <= 2000`; every run used the same number of steps; `len(LOG)` equals the number of runs, and every record has the seven keys; `HYPOTHESIS` and `DECISION_RULE` are non-empty strings defined before the first run (record `len(LOG) == 0` at that point); `decide(LOG)` equals the configuration that was scored; every seam value is `"train"`; the test-call protocol.

**Required answer-check assertions.** Every certificate item.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

import time

TOTAL_BUDGET = 2000
MAX_WIDTH = 64

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.